# 13 — **병변 검출기** (STEP 42)

## 왜 이게 남았나

네모를 받는 길이 **아홉 개** 닫혔습니다 (STEP 36~41):

| 길 | 결과 |
|---|---|
| 사용자가 네모 **크기** | ❌ 병변 크기와 무관 (상관 −0.05) |
| 앱 기본값 44% 조정 | ❌ 병변이 화면의 5.6~59.6% |
| `f320` 전환 (macro-F1 / 커버리지) | ❌ +0.021 / 라벨에서도 8.0% |
| 사용자가 **점 위치** (탭) | ❌ 0.111 > 0.10 · 병변 안 47.5% |
| 분류기를 **창 탐지기**로 | ❌ 둘 다 밴드 안 6.7% |
| 네모 잡음 증강 | ❌ 라벨 −0.084 |
| 네모 없이 학습 | ❌ 재현에서 뒤집힘 |
| 고정 배율 크롭 | ❌ −1.1%p (중심 오차와 상충) |

**전부 같은 곳에서 막힙니다** — 사람은 병변이 어디에 얼마나 크게 있는지
못 알려줍니다.

★ 그런데 **`bbox` 라벨이 36만 장** 있습니다. 사람에게 묻지 말고 **모델이 그
일을 직접 배우게** 하는 정공법인데, **시도조차 안 했습니다.**

⚠️ 위 "창 탐지기" 와 다릅니다 — 거기서는 **분류기를 빌려 썼습니다.**
네모를 뽑도록 배운 적이 없는 모델이었고 6.7% 였습니다. 여기서는 **직접
배웁니다.**

## 판정 (돌리기 전에 박아둔 것)

| 상수 | 뜻 |
|---|---|
| `DETECT_MIN_USABLE = 0.50` | 제안이 **배율·위치 밴드에 둘 다** 드는 비율 |
| `DETECT_MIN_COVERAGE_GAIN = 0.05` | 그 네모로 자른 크롭의 **계열 커버리지** 이득 |

⚠️ **둘 다** 넘어야 합니다. 네모가 좋아 보여도 크롭이 걸리면 중심 오차가
치명적이 될 수 있습니다 (STEP 41 에서 배운 것). **최종 판정은 항상 제품
지표(커버리지)로** 합니다.

## 🚨 이 노트북은 holdout 을 **안 엽니다**

`tests/test_holdout_discipline.py` 가 감시합니다. 판정은 val 로 합니다.

## 붙일 데이터

| Add input | 무엇 |
|---|---|
| `dogskin-detect` (Private) | `tools/make_detect_dataset.py` 가 만든 것 — **앱처럼 찍은 사진 + 정답 네모** |

⚠️ 크롭 데이터셋(`m2.5`·`f320`)은 **못 씁니다.** 병변을 가운데 놓고 자른
것이라 정답이 항상 한가운데입니다 — 모델이 "가운데" 만 외우면 됩니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
NAME   = "deeplearning_test"
# ⚠️ 브랜치를 "main" 으로 **못 박으면 안 됩니다.** 아래 reset --hard 가
#    작업 브랜치를 통째로 덮어써서, 방금 만든 코드가 사라진 채로 몇 시간을
#    돌게 됩니다. 이미 리포 안에서 돌고 있으면 **지금 브랜치를 그대로 씁니다.**
#    바꾸려면 환경변수:  export DOG_SKIN_BRANCH=main
# ★ 이 노트북이 사는 브랜치. **여기서 못 박지 않으면 "main" 을 받습니다.**
#    캐글/콜랩은 리포가 없는 상태로 시작해서 아래 _ROOT 탐색이 실패하고,
#    예전 기본값이 "main" 이었습니다. main 이 뒤처져 있으면 **셀은 최신인데
#    src/ 만 옛것**인 채로 돕니다 — 실제로 며칠 그랬습니다 (main 75445c0).
#    첫 셀은 그 상태에서도 "코드 버전 …" 을 태연히 찍습니다.
NB_BRANCH = "main"
BRANCH = os.environ.get("DOG_SKIN_BRANCH", "")
_cwd   = os.getcwd()

# ⚠️ "지금 리포 안인가" 를 **폴더 이름으로만** 보면 안 됩니다. 주피터에서
#    notebooks/*.ipynb 를 열면 cwd 가 `.../deeplearning_test/notebooks` 라
#    이름이 안 맞고, 그러면 **리포 안에 리포를 또 clone** 합니다
#    (실제로 런팟에서 .../notebooks/deeplearning_test 가 생겼습니다).
#    위로 거슬러 올라가며 **진짜 리포 루트**를 찾습니다.
_p = os.path.abspath(_cwd)
_ROOT = None
while True:
    if (os.path.isdir(os.path.join(_p, ".git"))
            and os.path.isfile(os.path.join(_p, "src", "env.py"))):
        _ROOT = _p
        break
    _up = os.path.dirname(_p)
    if _up == _p:
        break
    _p = _up

if _ROOT:
    DIR = _ROOT           # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
    if not BRANCH:
        BRANCH = subprocess.run(["git", "-C", DIR, "rev-parse", "--abbrev-ref", "HEAD"],
                                capture_output=True, text=True).stdout.strip() or "main"
else:
    # ⚠️ Kaggle 을 먼저 봅니다. Kaggle 이미지에도 /content 가 있어서
    #    /content 를 먼저 보면 Kaggle 세션인데 /content 에 clone 합니다.
    BASE = ("/kaggle/working" if os.path.isdir("/kaggle/working")
            else "/content" if os.path.isdir("/content") else _cwd)
    DIR = os.path.join(BASE, NAME)

BRANCH = BRANCH or NB_BRANCH

# ⚠️ 예전엔 fetch/reset 을 **둘 다 check=False** 로 불렀습니다. 실패해도 조용히
#    넘어가서, 캐글 클론이 **지워진 커밋(75445c0)에 붙박인 채 며칠을 돌았습니다.**
#    src/ 를 아무리 고쳐 푸시해도 안 실렸고, 첫 셀은 "코드 버전 …" 을 태연히
#    찍었습니다. 그 줄을 믿을 수 없다는 게 제일 나빴습니다.
#    → 이제 실패하면 **말하고, 클론을 지우고 다시 받습니다.**
#    (Kaggle Persistence 를 'Files' 로 켜두면 /kaggle/working 이 살아남아
#     낡은 클론이 계속 재사용됩니다 — 그 경우에도 여기서 복구됩니다.)
def _git(*args, cwd=None):
    return subprocess.run(["git", *args], capture_output=True, text=True, cwd=cwd)


def _fresh_clone(dst, branch):
    import shutil as _sh
    _sh.rmtree(dst, ignore_errors=True)
    r = _git("clone", "-b", branch, "--depth", "1", REPO, dst)
    if r.returncode != 0:
        raise RuntimeError("git clone 실패:\n" + (r.stderr or "")[-800:])


_need_clone = not os.path.isdir(os.path.join(DIR, ".git"))
if not _need_clone:
    # shallow clone 이라 origin/<브랜치> 대신 FETCH_HEAD 로 맞춥니다
    # (히스토리가 갈리면 origin/<브랜치> 가 옛 커밋을 가리킨 채 남습니다)
    r = _git("-C", DIR, "fetch", "--depth", "1", "origin", BRANCH)
    if r.returncode != 0:
        print("⚠️ git fetch 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
        _need_clone = True
    else:
        r = _git("-C", DIR, "reset", "--hard", "FETCH_HEAD")
        if r.returncode != 0:
            print("⚠️ git reset 실패 — 클론을 새로 받습니다\n   " + (r.stderr or "")[-300:])
            _need_clone = True

if _need_clone:
    _fresh_clone(DIR, BRANCH)

# ★ 정말 최신인지 **확인**합니다. 위가 다 성공해도 여기서 한 번 더 봅니다 —
#   "최신이라고 믿었는데 아니었다" 가 이 프로젝트에서 가장 비쌌던 실패입니다.
_local = _git("-C", DIR, "rev-parse", "HEAD").stdout.strip()
_remote = _git("-C", DIR, "ls-remote", REPO, f"refs/heads/{BRANCH}").stdout.split()
_remote = _remote[0] if _remote else ""
if _remote and _local and not _remote.startswith(_local[:8]) and not _local.startswith(_remote[:8]):
    print("\n" + "!" * 66)
    print(f"🚨 코드가 최신이 아닙니다 — 로컬 {_local[:8]} / 원격 {_remote[:8]}")
    print("   클론을 지우고 다시 받습니다.")
    print("!" * 66 + "\n")
    _fresh_clone(DIR, BRANCH)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", _git("-C", DIR, "log", "--oneline", "-1").stdout.strip())
print("브랜치      :", BRANCH,
      f"(원격 {_remote[:8]})" if _remote else "(원격 확인 실패)")
if BRANCH != NB_BRANCH:
    print(f"⚠️ 이 노트북이 만들어진 브랜치({NB_BRANCH})가 아닙니다 —")
    print("   src/ 가 셀보다 뒤처져 있을 수 있습니다. 아래 [nb] 줄을 꼭 보세요.")

# 패키지 설치는 **uv 로 통일**합니다 (pip 보다 훨씬 빠릅니다).
# ⚠️ Colab/Kaggle 이미지에는 uv 가 없어서, uv 자체만 pip 로 한 번 받습니다.
#    --system = 가상환경을 새로 만들지 않고 이미 있는 파이썬에 그대로 설치.
#    (torch/numpy/pandas 는 이미 깔려 있으므로 여기서 안 건드립니다)
# albumentations 는 import 할 때마다 PyPI 에 버전 확인 요청을 보냅니다.
# Kaggle 은 외부 네트워크가 막혀 있어 타임아웃(2초)만 기다리다 끝납니다 — 꺼둡니다.
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"

# ⚠️ 임대 GPU 이미지(런팟 등)의 파이썬은 **externally managed** 입니다 (PEP 668).
#    그냥 설치하면 첫 시도가 통째로 거부돼서, 재시도 로직이 있어도 무서운
#    에러 덩어리가 먼저 찍힙니다. 처음부터 허용해두면 그 소음이 없습니다.
#    Colab/Kaggle 에는 이 제약이 없어서 이 변수는 무해합니다.
os.environ["PIP_BREAK_SYSTEM_PACKAGES"] = "1"
os.environ["UV_BREAK_SYSTEM_PACKAGES"] = "1"

# ⚠️ Colab/Kaggle 에는 numpy·pandas·sklearn 이 이미 있지만 **임대 GPU 이미지엔
#    torch 만 있는 경우가 많습니다** (런팟에서 `No module named 'pandas'` 로
#    막혔습니다). 그렇다고 매번 다 깔면 Colab 에서 버전이 흔들리므로
#    **없는 것만** 깝니다.
_NEED = {                       # import 이름 → pip 이름
    "numpy": "numpy", "pandas": "pandas", "pyarrow": "pyarrow", "PIL": "Pillow",
    "sklearn": "scikit-learn", "cv2": "opencv-python-headless", "tqdm": "tqdm",
    "matplotlib": "matplotlib", "timm": "timm", "imagehash": "imagehash",
    "pytorch_grad_cam": "grad-cam", "albumentations": "albumentations",
}
import importlib.util as _ilu

_PKGS = [pip for mod, pip in _NEED.items() if _ilu.find_spec(mod) is None]
if _PKGS:
    print(f"[env] 없는 패키지 {len(_PKGS)}개를 깝니다: {_PKGS}")
else:
    print("[env] 필요한 패키지가 전부 있습니다 — 설치를 건너뜁니다")

# ⚠️ 일부 이미지(런팟 PyTorch 등)는 파이썬이 **externally managed** 라
#    (PEP 668) --system 설치를 거부합니다. Colab/Kaggle 에는 없는 문제라
#    처음엔 안 넣었다가 런팟에서 첫 셀이 바로 죽었습니다.
#    --break-system-packages 를 붙여 한 번 더 시도합니다.
def _install(args: list[str]) -> bool:
    return subprocess.run(args, check=False).returncode == 0


_ok = not _PKGS          # 깔 게 없으면 이미 성공입니다
if _PKGS and _install([sys.executable, "-m", "pip", "install", "-q", "uv"]):
    _base = [sys.executable, "-m", "uv", "pip", "install", "-q", "--system"]
    _ok = _install(_base + _PKGS)
    if not _ok:
        _ok = _install(_base + ["--break-system-packages"] + _PKGS)
if not _ok:
    print("[env] uv 로 설치하지 못해 pip 으로 대체합니다")
    _p = [sys.executable, "-m", "pip", "install", "-q"]
    if not _install(_p + _PKGS):
        _install(_p + ["--break-system-packages"] + _PKGS)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

MY_NOTEBOOK_VERSION = "2026-09-04.5"   # ★ 이 셀(=이 .ipynb)의 버전

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

# 환경 판정이 이상하면(예: Kaggle 인데 colab 이라고 나오면) 근거를 봅니다
if E.env != "local":
    env.diagnose()

# ⚠️ 노트북 셀은 git pull 로 갱신되지 않습니다 (src/ 만 최신이 됩니다).
#    낡은 .ipynb 를 몇 시간 돌리고 나서 알게 되면 늦으므로 지금 확인합니다.
from src.config import NOTEBOOK_VERSION as _repo_nb
if MY_NOTEBOOK_VERSION != _repo_nb:
    print("\n" + "!" * 62)
    print(f"⚠️ 이 노트북이 낡았습니다 — 내 셀 {MY_NOTEBOOK_VERSION} / 리포 {_repo_nb}")
    print("   src/ 는 최신이지만 **셀 내용은 예전 것**입니다.")
    print("   GitHub 에서 notebooks/*.ipynb 를 다시 받아 Import 하세요:")
    print("   Kaggle → File → Import Notebook / Colab → 파일 → 노트 업로드")
    print("!" * 62 + "\n")
else:
    print(f"[nb] 노트북 최신 ({_repo_nb})")


⚠️ **위 셀이 받는 브랜치를 확인하세요.** 이 노트북은 `main` 을 받습니다 —
`src/detect.py` 가 거기 있습니다. 다른 브랜치를 받으면
`ModuleNotFoundError: No module named 'src.detect'` 로 죽습니다
(실제로 한 번 그랬습니다).

## 1. 데이터 붙이기

`boxes.parquet` 과 `images/` 를 찾습니다. **못 찾으면 멈춥니다** — 조용히
물러서면 몇 시간을 버립니다.

In [ ]:
from pathlib import Path
import pandas as pd

# 캐글 입력 어디에 있든 찾습니다 (계정/데이터셋 이름이 사람마다 다릅니다)
cands = list(Path("/kaggle/input").rglob("boxes.parquet")) \
    if Path("/kaggle/input").exists() else []
cands += sorted(Path(".").glob("data/work/detect*/boxes.parquet"))
if not cands:
    raise SystemExit(
        "[X] boxes.parquet 을 못 찾았습니다.\n"
        "    Add input 으로 검출 데이터셋을 붙였는지 확인하세요.\n"
        f"    /kaggle/input 아래: {[p.name for p in Path('/kaggle/input').iterdir()] if Path('/kaggle/input').exists() else '(없음)'}")
BOXES = cands[0]
IMGS = BOXES.parent / "images"
if not IMGS.is_dir():
    raise SystemExit(f"[X] {IMGS} 가 없습니다 (캐글이 폴더를 한 겹 더 싸는 일이 있습니다)")
df = pd.read_parquet(BOXES)
print(f"■ {len(df):,}장 · 개체 {df['group'].nunique():,} · {BOXES.parent}")
print(f"■ 이미지 {sum(1 for _ in IMGS.iterdir()):,}개")
assert len(df) > 0

## 2. 개체 단위로 가릅니다

같은 개가 학습·평가에 갈라지면 **평가가 거짓말합니다.** 이 프로젝트에서
제일 먼저 못 박은 규칙입니다.

In [ ]:
import random
SEED = 17
gs = sorted(df["group"].astype(str).unique())
random.Random(SEED).shuffle(gs)
cut = int(len(gs) * 0.85)
tr_g, va_g = set(gs[:cut]), set(gs[cut:])
dtr = df[df["group"].astype(str).isin(tr_g)].reset_index(drop=True)
dva = df[df["group"].astype(str).isin(va_g)].reset_index(drop=True)
print(f"■ 학습 {len(dtr):,}장 / 평가 {len(dva):,}장")
print(f"■ 개체 {len(tr_g):,} / {len(va_g):,}  (겹침 {len(tr_g & va_g)})")
assert not (tr_g & va_g), "개체가 겹칩니다 — 평가가 거짓말합니다"

## 3. 학습

In [ ]:
import torch, time
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from src.detect import BoxHead, box_loss, to_xyxy, band_report, print_report
from src.config import CFG
from src import data as sdata

cfg = CFG(img_size=384)
tf = sdata.build_transforms(cfg, train=False)   # 검출은 기하 증강을 안 씁니다

class DS(Dataset):
    def __init__(self, d): self.d = d
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        with Image.open(IMGS / r["image"]) as im:
            x = tf(im.convert("RGB"))
        return x, torch.tensor([r.x1, r.y1, r.x2, r.y2], dtype=torch.float32)

dev = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS, BS, LR = 6, 24, 2e-4
net = BoxHead(pretrained=True, img_size=384).to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(dev == "cuda"))
ltr = DataLoader(DS(dtr), batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
lva = DataLoader(DS(dva), batch_size=BS, shuffle=False, num_workers=2)

def evaluate():
    net.eval(); P, T = [], []
    with torch.no_grad():
        for x, y in lva:
            P.append(to_xyxy(net(x.to(dev))).float().cpu()); T.append(y)
    return band_report(torch.cat(P).numpy(), torch.cat(T).numpy())

t0 = time.perf_counter()
best = None
for ep in range(EPOCHS):
    net.train(); tot = k = 0
    for x, y in ltr:
        x, y = x.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=(dev == "cuda")):
            loss = box_loss(net(x), y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        tot += float(loss.detach()); k += 1
    sched.step()
    rep = evaluate()
    print(f"ep{ep+1}  loss {tot/max(k,1):.4f}  둘다 {rep['both']:.1%}  "
          f"크기비 {rep['ratio_median']:.2f}  중심 {rep['off_median']:.3f}  "
          f"{(time.perf_counter()-t0)/60:.1f}분", flush=True)
    if best is None or rep["both"] > best["both"]:
        best = rep
        torch.save({"model": net.state_dict(), "report": rep},
                   "detect_best.pt")
print()
ok = print_report(best)

## 4. ⚠️ 여기서 멈춥니다

밴드 판정만 나왔습니다. **채택하려면 이 네모로 자른 크롭의 커버리지**가
`DETECT_MIN_COVERAGE_GAIN` 만큼 올라야 합니다 — 그건 2단계 모델이 필요하므로
**별도 노트북**에서 합니다.

⚠️ **여기서 설정을 바꿔 다시 돌리지 마세요.** 밴드가 미달이면 미달로 적습니다.

In [ ]:
import json
json.dump(best, open("detect_report.json", "w"), ensure_ascii=False, indent=1)
print("저장: detect_best.pt · detect_report.json")
print("\n⚠️ 이 노트북은 holdout 을 안 엽니다. 판정은 val 로 했습니다.")
print("⚠️ 밴드 통과는 절반입니다 — 커버리지가 안 오르면 채택 안 합니다.")